# Facilitator Answer Key — v2 Aligned Workflow

## Activity 2

In [ ]:
measurements=["red","green","blue","nir08","swir16","scl"]
mask_flags=[1,3,9,10]

## Activity 3

In [ ]:
aoi_gdf = gpd.read_file(AOI_PATH)
gdf = gpd.read_file(TRAINING_PATH, bbox=tuple(bbox))

## Activity 4

In [ ]:
datetime="2018-03/2018-07"
# collection: sentinel-2-c1-l2a
# source example cloud threshold: 25
# load bands: red, green, blue, nir08, swir16, scl

## Activity 5

In [ ]:
mask_flags=[1,3,9,10]
cloud_mask = ~data.scl.isin(mask_flags)
masked = data.where(cloud_mask)
non_zero = masked.where(masked != 0)
scaled = non_zero * 0.0001
scaled = scaled.clip(0,1)
scaled["ndvi"] = (scaled.nir08-scaled.red)/(scaled.nir08+scaled.red)
median = scaled.median("time").compute()

## Activity 6

In [ ]:
training = gdf.to_crs(median.odc.geobox.crs)
training_da = training.assign(x=training.geometry.x,y=training.geometry.y).to_xarray()
training_values=(median.sel(training_da[["x","y"]],method="nearest").squeeze().compute().to_pandas())
training_array=pd.concat([training["randomforest"],training_values],axis=1)
training_array=training_array.drop(columns=["y","x","spatial_ref"]).dropna()

## Activity 7

In [ ]:
classes=np.array(training_array)[:,0]
observations=np.array(training_array)[:,1:]
classifier=RandomForestClassifier(random_state=42)
model=classifier.fit(observations,classes)

## Activity 8

In [ ]:
stacked_arrays=median.to_array().stack(dims=["y","x"]).transpose()
predicted=model.predict(stacked_arrays)
array=predicted.reshape(len(median.y),len(median.x))
predicted_da=xr.DataArray(array,coords={"y":median.y,"x":median.x},dims=["y","x"]).astype("float32")
predicted_da.odc.write_cog("../outputs/cordia_prediction.tif")

## Activity 9

In [ ]:
cordia_pixels=int((predicted_da==CORDIA_CLASS).sum().item())
cordia_area_m2=cordia_pixels*pixel_area_m2
cordia_area_ha=cordia_area_m2/10000
valid_pixels=int(predicted_da.notnull().sum().item())
cordia_percent=(cordia_pixels/valid_pixels)*100